# Cross-vintage combined TOPSIS — 20260714 final grid (v1985 / v1990 / v1995)

One ranking over the configs shared by all three vintages, so a single "overall best"
M can be named on one metric basis (the per-vintage notebooks each prune to a *different*
metric set, so their TOPSIS objectives are not comparable).

Pipeline:
1. Load each vintage's final `dh_results.parquet`, aggregate to one row per config (OOS).
2. Apply the **same filter** as the per-vintage notebooks (non-finite cull + calibration gate).
3. Run the trimmed-ratio sweep per vintage (needed for the `trim_n=*` calib metrics).
4. Keep only configs that survive the filter in **all three** vintages (matched by the
   13-field config tuple — `mid` differs per vintage because it encodes the data-derived
   `threshold_rate`, so matching is on the config, not the hash).
5. Paste the vintage year into every metric name → one wide row per config.
6. Pick metrics (the long editable menu) → Kendall τ → prune → **combined TOPSIS**.


In [ ]:
import json
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idd_tc_mortality.cache import model_id as compute_model_id
from idd_tc_mortality.select.model_selection import (
    CONFIG_COLS,
    prepare_rankings_df,
    topsis_rank,
    kendall_tau_heatmap,
    prune_redundant_metrics,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
pd.options.display.max_columns = 120
pd.options.display.width = 200

## Config

`RATIO_BOUNDS` is **your filter setting** — the same asymmetric calibration gate the
per-vintage notebooks use. Edit it (and `N_SWEEP` / `PRUNE_THRESHOLD`) here.

In [ ]:
ROOT     = Path('/mnt/team/idd/pub/idd_tc_mortality')
VINTAGES = ['1985', '1990', '1995']          # -> 20260714_v<Y>_final
YEARS    = {v: v for v in VINTAGES}          # suffix pasted into metric names

RATIO_BOUNDS    = (0.05, 1.1)                # calibration gate on full_pred_obs_ratio_oos
N_SWEEP         = [0, 5, 10, 25]             # trimmed-ratio drop-top-N levels
PRUNE_THRESHOLD = 0.7                        # |Kendall tau| above which a metric is redundant
N_WORKERS       = 12                         # process pool for the trim sweep

## Per-vintage load + filter + trim sweep

`load_vintage` reproduces the per-vintage final notebook exactly: prepare → non-finite
cull → calibration gate → attach `mid` → trimmed-ratio sweep (read each config's 6
prediction parquets once, per-storm table sliced per N, fanned over a process pool).

In [ ]:
# --- module-level helpers (globals set per vintage before each pool is forked) ---
MODEL_PRED_DIR = None   # set by load_vintage
_OBS = None             # this vintage's input.parquet

def _join_input(pred_df):
    if len(pred_df) != len(_OBS):
        raise ValueError(f'prediction rows {len(pred_df)} != input rows {len(_OBS)}')
    out = _OBS.copy()
    out['predicted_rate'] = pred_df['predicted_rate'].values
    return out

def load_is_pred(mid):
    p = MODEL_PRED_DIR / f'{mid}_insample_predictions.parquet'
    return _join_input(pd.read_parquet(p)) if p.exists() else pd.DataFrame()

def load_oos_pred(mid):
    frames = []
    for seed in range(5):
        p = MODEL_PRED_DIR / f'{mid}_oos_seed{seed}_predictions.parquet'
        if p.exists():
            j = _join_input(pd.read_parquet(p)); j['seed'] = seed; frames.append(j)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _per_storm(pred_df):
    pred_df = pred_df.assign(pred_deaths=pred_df['predicted_rate'] * pred_df['exposed'])
    return (pred_df.groupby('storm_id', as_index=False)
            .agg(obs=('deaths', 'sum'), pred=('pred_deaths', 'sum'))
            .sort_values('obs', ascending=False))

def _trimmed(per_storm, n):
    keep = per_storm.iloc[n:]; denom = keep['obs'].sum()
    return float(keep['pred'].sum() / denom) if denom > 0 else float('nan')

def trimmed_ratios_one_config(mid):
    is_df = load_is_pred(mid)
    ps_is = None if is_df.empty else _per_storm(is_df)
    oos = load_oos_pred(mid)
    ps_seeds = [] if (oos.empty or 'predicted_rate' not in oos.columns) else [
        _per_storm(sdf) for _, sdf in oos.groupby('seed')]
    rows = []
    for n in N_SWEEP:
        per_seed = [_trimmed(ps, n) for ps in ps_seeds]
        rows.append({'mid': mid, 'drop_top_n': n,
                     'ratio_mean':   float(np.nanmean(per_seed)) if per_seed else float('nan'),
                     'ratio_median': float(np.nanmedian(per_seed)) if per_seed else float('nan'),
                     'ratio_is':     _trimmed(ps_is, n) if ps_is is not None else float('nan')})
    return rows

def _build_spec_lookup(manifest):
    lut = {}
    for spec in manifest.values():
        if not isinstance(spec, dict) or spec.get('fold_tag', 'is') != 'is':
            continue
        cov_str = json.dumps(spec.get('covariate_combo', {}), sort_keys=True)
        lut[(spec.get('component'), spec.get('threshold_quantile'),
             spec.get('family'), spec.get('exposure_mode'), cov_str)] = spec
    return lut

def _row_to_mid(row, lut):
    q = row['threshold_quantile']
    s1 = lut[('s1',   None, row['s1_family'],   row['s1_exposure_mode'],   row['s1_cov'])]
    s2 = lut[('s2',   q,    row['s2_family'],   row['s2_exposure_mode'],   row['s2_cov'])]
    bk = lut[('bulk', q,    row['bulk_family'], row['bulk_exposure_mode'], row['bulk_cov'])]
    tl = lut[('tail', q,    row['tail_family'], row['tail_exposure_mode'], row['tail_cov'])]
    return compute_model_id(s1, s2, bk, tl)

def load_vintage(v):
    global MODEL_PRED_DIR, _OBS
    eval_dir = ROOT / '02-evaluate' / f'20260714_v{v}_final'
    MODEL_PRED_DIR = eval_dir / 'model_predictions'
    _OBS = pd.read_parquet(ROOT / '00-data' / f'20260714_v{v}' / 'input.parquet')

    dh = pd.read_parquet(eval_dir / 'dh_results.parquet')
    df = prepare_rankings_df(dh, subset='oos')
    n0 = len(df)

    oos_numeric = [c for c in df.columns
                   if c.endswith('_oos') and pd.api.types.is_numeric_dtype(df[c])]
    df = df[np.isfinite(df[oos_numeric].to_numpy()).all(axis=1)].copy()
    df = df[df['full_pred_obs_ratio_oos'].between(*RATIO_BOUNDS)].copy()

    manifest = json.loads((eval_dir / 'manifest.json').read_text())
    lut = _build_spec_lookup(manifest)
    df['mid'] = df.apply(lambda r: _row_to_mid(r, lut), axis=1)

    mids = df['mid'].tolist()
    records = []
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        for rows in pool.map(trimmed_ratios_one_config, mids, chunksize=8):
            records.extend(rows)
    recs = pd.DataFrame(records)
    for stat, col in (('oos_mean', 'ratio_mean'), ('oos_median', 'ratio_median'), ('is', 'ratio_is')):
        piv = recs.pivot(index='mid', columns='drop_top_n', values=col)
        piv.columns = [f'trim_n={n}_{stat}' for n in piv.columns]
        df = df.merge(piv, left_on='mid', right_index=True, how='left')

    print(f'v{v}: {n0:,} -> {len(df):,} configs after filter (+ {df.filter(like="trim_n=").shape[1]} trim cols)')
    return df

In [ ]:
frames = {v: load_vintage(v) for v in VINTAGES}

## Combine: keep configs present in all three, paste the year into each metric

Match on the 13-field config tuple (`CONFIG_COLS`). The result is one wide row per
shared config, with every OOS/trim metric suffixed by vintage year, plus the config
columns and each vintage's `mid` (the focus-model input for the draws stage).

In [ ]:
def _key(df):
    return list(map(tuple, df[CONFIG_COLS].astype(str).to_numpy()))

# configs surviving the filter in ALL three vintages
common = None
for v, f in frames.items():
    ks = set(_key(f))
    common = ks if common is None else (common & ks)
print(f'configs shared (post-filter) by all three vintages: {len(common):,}')

# metric columns to carry (per-vintage numeric OOS metrics + trim columns)
def _metric_cols(df):
    return [c for c in df.columns
            if (c.endswith('_oos') and pd.api.types.is_numeric_dtype(df[c])) or c.startswith('trim_n=')]

wide = None
mids_by_v = {}
for v, f in frames.items():
    f = f.copy()
    f['__key'] = _key(f)
    f = f[f['__key'].isin(common)].drop_duplicates('__key').set_index('__key')
    mids_by_v[v] = f['mid']
    sub = f[_metric_cols(f)].add_suffix(f'_{v}')
    wide = sub if wide is None else wide.join(sub, how='inner')

# reattach the 13 config columns (identical across vintages for a given key)
ref = frames[VINTAGES[0]].copy(); ref['__key'] = _key(ref)
ref = ref[ref['__key'].isin(common)].drop_duplicates('__key').set_index('__key')
df_wide = ref[CONFIG_COLS].join(wide, how='inner')
for v in VINTAGES:
    df_wide[f'mid_{v}'] = mids_by_v[v]

print(f'df_wide: {df_wide.shape[0]:,} configs x {df_wide.shape[1]} cols '
      f'({len([c for c in df_wide.columns if c.endswith(tuple("_"+v for v in VINTAGES)) ])} year-suffixed metric cols)')
df_wide.head(3)

## Metrics to include (edit me) — including calib

Toggle base metrics on/off below. Each active base metric is expanded to one column
per vintage (`<metric>_<year>`), so a metric enters the combined ranking on equal
footing across all three vintages. `'calib'` = closer to 1.0 is better (handled by
the ranking API); `'higher'`/`'lower'` as usual.

In [ ]:
# direction per BASE metric name (year suffix added automatically below)
BASE_METRICS = {
    # ---- S1 (binary: P(deaths >= 1)) ----
    's1_auroc_oos':               'higher',
    # 's1_brier_oos':             'lower',
    # 's1_fpr_oos':               'lower',
    # 's1_fnr_oos':               'lower',

    # ---- S2 (binary: P(rate >= thresh | deaths >= 1)) ----
    # 's2_auroc_oos':             'higher',
    # 's2_brier_oos':             'lower',

    # ---- Bulk (regression on rate < thresh) ----
    'bulk_mae_rate_oos':          'lower',
    # 'bulk_rmse_rate_oos':       'lower',
    # 'bulk_cor_rate_oos':        'higher',

    # ---- Tail (regression on rate >= thresh) ----
    # 'tail_mae_rate_oos':        'lower',
    # 'tail_rmse_rate_oos':       'lower',
    # 'tail_cor_rate_oos':        'higher',

    # ---- Full (unconditional E[rate], applies to every row) ----
    'full_mae_rate_oos':          'lower',
    # 'full_rmse_rate_oos':       'lower',
    # 'full_cor_rate_oos':        'higher',
    # 'full_zero_acc_oos':        'higher',
    'full_pred_obs_ratio_oos':    'calib',
    # 'full_coverage_rate_5_oos':   'higher',
    # 'full_coverage_rate_10_oos':  'higher',
    'full_coverage_rate_20_oos':  'higher',
    # 'full_coverage_count_5_oos':  'higher',
    # 'full_coverage_count_10_oos': 'higher',
    'full_coverage_count_20_oos': 'higher',

    # ---- Forward (stage-conditional aggregate) ----
    # 'fwd_mae_rate_oos':         'lower',
    # 'fwd_pred_obs_ratio_oos':   'calib',
    # 'fwd_coverage_rate_5_oos':    'higher',
    # 'fwd_coverage_rate_10_oos':   'higher',
    'fwd_coverage_rate_20_oos':   'higher',
    # 'fwd_coverage_count_5_oos':   'higher',
    # 'fwd_coverage_count_10_oos':  'higher',
    'fwd_coverage_count_20_oos':  'higher',

    # ---- Trimmed calibration ratios (drop-top-N mega-storms) ----
    # 'trim_n=5_oos_median':      'calib',
    'trim_n=10_oos_median':       'calib',
    # 'trim_n=25_oos_median':     'calib',
    'trim_n=10_is':               'calib',
    # 'trim_n=5_is':              'calib',
    # 'trim_n=25_is':             'calib',
}

In [ ]:
# expand base metrics to year-suffixed columns present in df_wide
metrics_to_use = {}
for base, direction in BASE_METRICS.items():
    for v in VINTAGES:
        col = f'{base}_{v}'
        if col in df_wide.columns:
            metrics_to_use[col] = direction

calib_present   = [k for k, d in metrics_to_use.items() if d == 'calib']
metrics_present = {k: d for k, d in metrics_to_use.items()}   # keep calib in the dict too

print(f'Active rank metrics ({len(metrics_present)}) across {len(VINTAGES)} vintages:')
for m, d in metrics_present.items():
    print(f'  {m:38s}  {d}')
missing = [f'{b}_{v}' for b in BASE_METRICS for v in VINTAGES
           if f'{b}_{v}' not in df_wide.columns]
if missing:
    print(f'\n[note] {len(missing)} requested (metric x vintage) not in df_wide: {missing}')

## Kendall τ → prune redundant metrics

Same as the per-vintage notebooks, but now the correlation is across the combined
(year-suffixed) metric set — so highly-redundant metrics (including the same base
metric across vintages that move together) get pruned once, on one basis.

In [ ]:
print(f'Kendall correlations on {len(df_wide):,} shared configs x {len(metrics_present)} metrics')
tau_df = kendall_tau_heatmap(df_wide, metrics_present, calib_present)
metrics_present, prune_log = prune_redundant_metrics(tau_df, metrics_present, threshold=PRUNE_THRESHOLD)
tau_df = kendall_tau_heatmap(df_wide, metrics_present, calib_present)
plt.show()
print(f'\nkept {len(metrics_present)} metrics after prune')

## Combined TOPSIS

One ranking over all shared configs. The top rows are candidate universal M's; each
row's `mid_<year>` is the focus-model id to feed the draws stage for that vintage.

In [ ]:
topsis_df, _ = topsis_rank(df_wide, metrics_present, calib_present, verbose=False)

show = (['combined_topsis_rank', 'topsis_score', 'threshold_quantile',
         'tail_family', 'tail_exposure_mode', 'bulk_exposure_mode',
         's1_cov', 's2_cov', 'bulk_cov', 'tail_cov']
        + [f'mid_{v}' for v in VINTAGES])
topsis_df = topsis_df.rename(columns={'topsis_rank': 'combined_topsis_rank'})
topsis_df = topsis_df.sort_values('combined_topsis_rank')
topsis_df[show].head(15)

In [ ]:
# The single overall-best M (combined rank 1), full 13-field formulation + per-vintage mids
best = topsis_df.iloc[0]
print(f'Combined-best M  (topsis_score={best["topsis_score"]:.4f})')
for c in CONFIG_COLS:
    print(f'  {c:22s} = {best[c]}')
for v in VINTAGES:
    print(f'  mid_{v}                = {best[f"mid_{v}"]}')